In [15]:
import os
import streamlit as st
import pickle
import time
import langchain
from langchain import OpenAI
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.chains.qa_with_sources.loading import load_qa_with_sources_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import UnstructuredURLLoader
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings



In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [10]:
#initialize the LLM
llm = OpenAI(temperature=0.9, max_tokens=500)

loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/wall-street-turns-crypto-risk-into-a-530-million-complex-trade-13728686.html",
    "https://www.moneycontrol.com/automobile/auto-makers-must-introspect-on-ev-commitment-to-meet-2030-goals-tata-motors-shailesh-chandra-article-13739940.html" ])

data = loaders.load()
len(data)

2

In [11]:
data[0].page_content

"English\n\nHindi\n\nGujarati\n\nSpecials\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\nGo Ad-Free\n\nMy Alerts\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_BUSINESS_AS/MC_ENG_ROS_NWS_BUS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nHomeNewsBusinessWall Street turns crypto risk into a $530 million complex trade\n\nTrending Topics\n\nSensex Live\n\nMarc Technocrats Share Price\n\nShyam Dhani Industries IPO\n\nKSH International Shares\n\nGold Rate Today\n\nWall Street turns cry

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

#as data is of type document we can directly use split_document over split_text in order to get the chunks.
docs = text_splitter.split_documents(data)
len(docs)

23

In [13]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/wall-street-turns-crypto-risk-into-a-530-million-complex-trade-13728686.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\nGo Ad-Free\n\nMy Alerts\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_BUSINESS_AS/MC_ENG_ROS_NWS_BUS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nHomeNewsBusinessWall Street turns crypto risk into a $530 million complex trade\n

In [ ]:
embedding = OpenAIEmbeddings()

vectorindex_openai = FAISS.from_documents(docs, embedding)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [24]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

vectorindex = FAISS.from_documents(docs, embedding)


In [26]:
#storing vector index create in local

file_path = "vector_index.pkl"
with open(file_path, "wb") as f:
    pickle.dump(vectorindex,f)

In [27]:
if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        vectorind =  pickle.load(f)

In [ ]:
chain = RetrievalQAWithSourcesChain(llm=llm, retriever=vectorind.as_retriever())
chain

In [ ]:
query = "What is price of tiago iCNG?"
langchain.debug = True

chain({"question": query}, return_only_outputs=True)